In [9]:
import os, zipfile

path = "/content/archive.zip"

print("File exists:", os.path.exists(path))

if os.path.exists(path):
    print("File size:", os.path.getsize(path))
    print("Valid ZIP:", zipfile.is_zipfile(path))

File exists: True
File size: 253755392
Valid ZIP: False


In [10]:

import os, zipfile, cv2, numpy as np

zip_path = "/content/archive.zip"
extract_path = "/content/signature_dataset"

# Check ZIP
if not zipfile.is_zipfile(zip_path):
    raise ValueError("Please upload a valid archive.zip!")

# Extract dataset
os.makedirs(extract_path, exist_ok=True)
with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_path)

# Find dataset folders
genuine_path = forged_path = None

for root, dirs, files in os.walk(extract_path):
    if os.path.basename(root) == "full_org":
        genuine_path = root
    if os.path.basename(root) == "full_forg":
        forged_path = root

if not genuine_path or not forged_path:
    raise FileNotFoundError("full_org or full_forg folder not found!")

# Load and preprocess images
IMG_SIZE = 128

def load_images(folder):
    images = []
    for file in os.listdir(folder):
        path = os.path.join(folder, file)
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if img is not None:
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            img = img.astype(np.float32) / 127.5 - 1
            images.append(img)
    return np.array(images, dtype=np.float32)

genuine = load_images(genuine_path)
forged = load_images(forged_path)

assert len(genuine) > 0 and len(forged) > 0

print("Genuine images:", len(genuine))
print("Forged images:", len(forged))
print("Step 1 completed!")


Genuine images: 1320
Forged images: 1320
Step 1 completed!


In [11]:

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import random

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

class SignatureDataset(Dataset):
    def __len__(self):
        return max(len(genuine), len(forged))

    def __getitem__(self, i):
        a = torch.tensor(genuine[i % len(genuine)]).unsqueeze(0)
        b = torch.tensor(forged[random.randrange(len(forged))]).unsqueeze(0)
        return a, b

loader = DataLoader(SignatureDataset(), batch_size=32, shuffle=True)

class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1,32,7,padding=3), nn.ReLU(),
            nn.Conv2d(32,64,4,2,1), nn.ReLU(),
            nn.Conv2d(64,128,4,2,1), nn.ReLU(),
            nn.ConvTranspose2d(128,64,4,2,1), nn.ReLU(),
            nn.ConvTranspose2d(64,32,4,2,1), nn.ReLU(),
            nn.Conv2d(32,1,7,padding=3), nn.Tanh()
        )
    def forward(self,x):
        return self.net(x)

class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1,32,4,2,1), nn.LeakyReLU(0.2),
            nn.Conv2d(32,64,4,2,1), nn.BatchNorm2d(64), nn.LeakyReLU(0.2),
            nn.Conv2d(64,128,4,2,1), nn.BatchNorm2d(128), nn.LeakyReLU(0.2),
            nn.Conv2d(128,1,3,padding=1)
        )
    def forward(self,x):
        return self.net(x)

G_AB, G_BA = Generator().to(device), Generator().to(device)
D_A, D_B = Discriminator().to(device), Discriminator().to(device)

mse, l1 = nn.MSELoss(), nn.L1Loss()

optG = torch.optim.Adam(
    list(G_AB.parameters()) + list(G_BA.parameters()),
    lr=0.0002, betas=(0.5,0.999)
)
optA = torch.optim.Adam(D_A.parameters(), lr=0.0002)
optB = torch.optim.Adam(D_B.parameters(), lr=0.0002)

# Fast training: one epoch
for epoch in range(1):
    for a,b in loader:
        a,b = a.to(device), b.to(device)

        optG.zero_grad()
        fakeB, fakeA = G_AB(a), G_BA(b)
        recA, recB = G_BA(fakeB), G_AB(fakeA)

        predA, predB = D_A(fakeA), D_B(fakeB)
        lossG = (
            mse(predA,torch.ones_like(predA)) +
            mse(predB,torch.ones_like(predB)) +
            10*(l1(recA,a)+l1(recB,b))
        )
        lossG.backward()
        optG.step()

        optA.zero_grad()
        realA, genA = D_A(a), D_A(fakeA.detach())
        lossA = (mse(realA,torch.ones_like(realA)) +
                 mse(genA,torch.zeros_like(genA))) / 2
        lossA.backward()
        optA.step()

        optB.zero_grad()
        realB, genB = D_B(b), D_B(fakeB.detach())
        lossB = (mse(realB,torch.ones_like(realB)) +
                 mse(genB,torch.zeros_like(genB))) / 2
        lossB.backward()
        optB.step()

print("Step 2 completed!")
print("CycleGAN training finished.")


Device: cuda
Step 2 completed!
CycleGAN training finished.


In [12]:

from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

# Generate synthetic research samples
G_AB.eval()

with torch.no_grad():
    sample = torch.tensor(genuine[:4]).unsqueeze(1).to(device)
    generated = G_AB(sample)

print("Generated samples:", generated.shape)

# Prepare dataset
X = np.concatenate([genuine, forged])
y = np.concatenate([
    np.zeros(len(genuine), dtype=np.int64),
    np.ones(len(forged), dtype=np.int64)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

train_loader = DataLoader(
    TensorDataset(
        torch.tensor(X_train).unsqueeze(1),
        torch.tensor(y_train)
    ),
    batch_size=32,
    shuffle=True
)

# CNN detector
class Detector(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1,32,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64,128,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(128*16*16,128), nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128,2)
        )

    def forward(self,x):
        return self.net(x)

detector = Detector().to(device)
optimizer = torch.optim.Adam(detector.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

# Fast training
for epoch in range(3):
    detector.train()
    total_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        output = detector(images)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/3 | Loss: {total_loss/len(train_loader):.4f}")

print("Step 3 completed!")
print("Forgery detector training finished!")


Generated samples: torch.Size([4, 1, 128, 128])
Epoch 1/3 | Loss: 0.6421
Epoch 2/3 | Loss: 0.2109
Epoch 3/3 | Loss: 0.1125
Step 3 completed!
Forgery detector training finished!


In [13]:

import os, cv2, torch, shutil
from sklearn.metrics import accuracy_score, classification_report

# Evaluate detector
detector.eval()

with torch.no_grad():
    test_tensor = torch.tensor(X_test).unsqueeze(1).to(device)
    predictions = detector(test_tensor).argmax(1).cpu().numpy()

accuracy = accuracy_score(y_test, predictions)

print("===== FINAL RESULTS =====")
print("Detection Accuracy:", round(accuracy * 100, 2), "%")

print(classification_report(
    y_test, predictions,
    target_names=["Genuine", "Forged"],
    zero_division=0
))

# Save trained models
os.makedirs("/content/models", exist_ok=True)

torch.save(G_AB.state_dict(), "/content/models/generator.pth")
torch.save(G_BA.state_dict(), "/content/models/reverse_generator.pth")
torch.save(detector.state_dict(), "/content/models/signature_detector.pth")

# Create ZIP file
shutil.make_archive(
    "/content/signature_project_models",
    "zip",
    "/content/models"
)

print("Step 4 completed!")
print("All models saved successfully!")

# Download models
from google.colab import files
files.download("/content/signature_project_models.zip")


===== FINAL RESULTS =====
Detection Accuracy: 99.81 %
              precision    recall  f1-score   support

     Genuine       1.00      1.00      1.00       264
      Forged       1.00      1.00      1.00       264

    accuracy                           1.00       528
   macro avg       1.00      1.00      1.00       528
weighted avg       1.00      1.00      1.00       528

Step 4 completed!
All models saved successfully!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>